# Unicode 正規化工具：清除幽靈字元

## 對應專欄：〈當「字」不僅止於字〉

本筆記本實作 **Unicode 正規化**（NFC / NFKC），用於處理文本中的「CJK 相容表意文字」（
中日韓相容表意文字，U+F900–U+FAFF）。這些「幽靈字元」在視覺上與標準漢字幾乎完全相同，
但會導致翻譯記憶庫（TM）檢索失效、模糊匹配率異常降低，以及其他在地化工程上的技術問題。

### NFC vs. NFKC 快速對照

| 形式 | 行為 | 適用場景 |
|------|------|----------|
| **NFC**（標準合成） | 僅將相容表意文字還原為標準字，**保留**全形字母、帶圈數字、連字等排版字元 | ✅ **建議預設使用**，安全修正幽靈字元 |
| **NFKC**（相容合成） | 全面拆解所有相容字元：全形→半形、帶圈數字→純數字、連字→拆解 | ⚠️ 僅在確認排版不會受影響時使用 |

> **實務口訣**：「除錯用 NFC，除非你想破壞排版，否則永遠別碰 NFKC。」

---


In [ ]:
# @title ❶ 設定參數（請在此輸入）
# @markdown 在下方欄位中填入或選擇參數：

input_file = "input.txt"  # @param {"type":"string"}
output_file = "output_normalized.txt"  # @param {"type":"string"}
norm_form = "NFC"  # @param ["NFC", "NFKC"]
file_encoding = "utf-8"  # @param ["utf-8", "utf-16", "big5", "gbk", "utf-8-sig"]

print(f"輸入檔案：{input_file}")
print(f"輸出檔案：{output_file}")
print(f"正規化形式：{norm_form}")
print(f"檔案編碼：{file_encoding}")


In [ ]:
# @title ❷ 載入檔案
# @markdown 若檔案已存在於 Colab 環境（例如已掛載 Google Drive），則直接讀取；否則請上傳。

import os
from google.colab import files

uploaded = None
if os.path.exists(input_file):
    print(f"✅ 找到檔案：{input_file}")
else:
    print(f"📤 檔案「{input_file}」不存在於當前目錄。")
    print("請從本機上傳檔案：")
    uploaded = files.upload()
    if uploaded:
        input_file = list(uploaded.keys())[0]
        print(f"✅ 已上傳：{input_file}")
    else:
        raise FileNotFoundError("未上傳任何檔案，程序終止。")

with open(input_file, "r", encoding=file_encoding) as f:
    text = f.read()

print(f"檔案大小：{len(text)} 字元")


In [ ]:
# @title ❸ 執行 Unicode 正規化

import unicodedata

normalized = unicodedata.normalize(norm_form, text)

with open(output_file, "w", encoding=file_encoding) as f:
    f.write(normalized)

print(f"✅ 正規化完成！")
print(f"   輸入：{input_file}")
print(f"   輸出：{output_file}")
print(f"   形式：{norm_form}")
print(f"   編碼：{file_encoding}")
print(f"   原始長度：{len(text)} 字元")
print(f"   正規化後：{len(normalized)} 字元")


In [ ]:
# @title ❹ 變更統計與內容預覽
# @markdown 以下顯示哪些字元被正規化取代，以及前後文對照。

import unicodedata

# 逐字元比對，找出被變更的字元
changes = []
for i, (orig, normed) in enumerate(zip(text, normalized)):
    if orig != normed:
        changes.append((i, orig, normed, unicodedata.name(orig, "?"), unicodedata.name(normed, "?")))

if changes:
    print(f"🔍 共發現 {len(changes)} 個字元被取代：")
    print(f"\n{"位置":>8} | {"原始":>8} | {"原始碼位":>12} | {"正規化後":>8} | {"正規化碼位":>12}")
    print("-" * 70)
    for idx, orig, normed, _, _ in changes[:30]:
        print(f"{idx:>8} | {orig:>8} | {hex(ord(orig)):>12} | {normed:>8} | {hex(ord(normed)):>12}")
    if len(changes) > 30:
        print(f"... 尚有 {len(changes) - 30} 個變更未顯示")
else:
    print("✅ 未發現任何被正規化取代的字元。")

print(f"\n=== 原始文字預覽（前 300 字元）===")
print(text[:300])
print(f"\n=== 正規化後預覽（前 300 字元）===")
print(normalized[:300])


In [ ]:
# @title ❺ 下載正規化結果
# @markdown 點擊下方按鈕下載正規化後的檔案。

from google.colab import files
print(f"正在準備下載：{output_file}")
files.download(output_file)
